# 🧱 Lab: Document Data Extractor

**Module 3: Prompt Engineering** | **Duration: ~1 hour** | **Type: Wall Lab**

## Learning Objectives
1. Extract structured data from unstructured text
2. Compare zero-shot vs few-shot extraction
3. Use JSON mode for guaranteed valid output
4. Validate extracted data with Pydantic

## Concepts: structured-output, few-shot, json-mode, output-parsing

In [8]:
import json
import os
from pathlib import Path
from openai import OpenAI
from pydantic import BaseModel
from typing import List, Optional
from dotenv import load_dotenv
from IPython.display import Markdown, display

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

load_dotenv()
client = OpenAI()

# Load all invoice files from data folder
DATA_DIR = Path('data')
INVOICES = {}

for file in sorted(DATA_DIR.glob('invoice_*.txt')):
    INVOICES[file.stem] = file.read_text()

# Use first invoice as sample for initial examples
SAMPLE_INVOICE = list(INVOICES.values())[0] if INVOICES else ""

print(f'✓ Loaded {len(INVOICES)} invoices from data folder:')
for name in INVOICES.keys():
    print(f'  - {name}')

print(f'\n📄 Sample invoice preview (first one):')
print(SAMPLE_INVOICE[:300] + '...' if len(SAMPLE_INVOICE) > 500 else SAMPLE_INVOICE)

✓ Loaded 5 invoices from data folder:
  - invoice_002_simple_receipt
  - invoice_003_email_style
  - invoice_004_european_format
  - invoice_005_handwritten_style
  - invoice_006_minimal

📄 Sample invoice preview (first one):
RECEIPT

CloudHost Services
------------------
Receipt #: REC-8847
Date: March 22, 2024

Customer: DataFlow Analytics

Services Rendered:
* Cloud Hosting (Monthly)     $299.00
* Extra Storage 500GB         $45.00
* SSL Certificate             $12.00
* Priority Support            $50.00

------------------------
TOTAL PAID: $406.00
------------------------

Payment Method: Credit Card (****4521)
Transaction ID: TXN-20240322-8847

Questions? support@cloudhost.io



## 2. Zero-Shot Extraction (~10 min)

Ask the LLM to extract data without any examples.

In [2]:
def zero_shot_extract(doc, fields):
    """Extract fields from document using zero-shot prompting."""
    prompt = f'''Extract the following fields from the document: {fields}

Document:
{doc}

Return the extracted data as JSON.'''
    
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0
    )
    return resp.choices[0].message.content

# Test zero-shot extraction
result = zero_shot_extract(SAMPLE_INVOICE, ['invoice_number', 'vendor', 'total'])
print('Zero-shot extraction result:')
print(result)

Zero-shot extraction result:
```json
{
  "invoice_number": "INV-2024-001",
  "vendor": "TechCorp Solutions",
  "total": "$6,200"
}
```


## 3. Few-Shot Extraction (~15 min)

Provide examples to guide the extraction format.

In [3]:
def few_shot_extract(doc, examples):
    """Extract data using few-shot examples."""
    # Format examples
    examples_text = '\n\n'.join([
        f'Document: {ex["doc"]}\nExtracted: {json.dumps(ex["result"])}'
        for ex in examples
    ])
    
    prompt = f'''Extract structured data from documents.

Examples:
{examples_text}

Now extract from this document:
{doc}

Extracted:'''
    
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0
    )
    return resp.choices[0].message.content

# Define few-shot examples
examples = [
    {
        'doc': 'Invoice #A1 from CompanyX to CustomerY, Total $100',
        'result': {'invoice_number': 'A1', 'vendor': 'CompanyX', 'customer': 'CustomerY', 'total': 100}
    },
    {
        'doc': 'INV-99 Seller: MegaCorp Buyer: SmallBiz Amount: $500',
        'result': {'invoice_number': 'INV-99', 'vendor': 'MegaCorp', 'customer': 'SmallBiz', 'total': 500}
    }
]

result = few_shot_extract(SAMPLE_INVOICE, examples)
print('Few-shot extraction result:')
print(result)

Few-shot extraction result:
{"invoice_number": "INV-2024-001", "vendor": "TechCorp Solutions", "customer": "Acme Inc", "total": 6200}


## 4. JSON Mode (~10 min)

Force the LLM to return valid JSON.

In [4]:
def json_extract(doc, schema_description):
    """Extract data with guaranteed JSON output."""
    prompt = f'''Extract data from the document according to this schema: {schema_description}

Document:
{doc}

Return only valid JSON with the extracted fields.'''
    
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'},
        temperature=0
    )
    return json.loads(resp.choices[0].message.content)

# Test JSON mode
result = json_extract(SAMPLE_INVOICE, 'invoice_number (string), vendor (string), customer (string), total (number)')
print('JSON mode extraction result:')
print(json.dumps(result, indent=2))
print(f'\nType of result: {type(result)}')

JSON mode extraction result:
{
  "invoice_number": "INV-2024-001",
  "vendor": "TechCorp Solutions",
  "customer": "Acme Inc",
  "total": 6200
}

Type of result: <class 'dict'>


## 5. Pydantic Validation (~15 min)

Use Pydantic models for type validation and data integrity.

In [5]:
# Define Pydantic model for invoice
class Invoice(BaseModel):
    invoice_number: str
    vendor: str
    customer: str
    total: float
    date: Optional[str] = None

def extract_pydantic(doc, model):
    """Extract and validate data using Pydantic model."""
    schema = model.model_json_schema()
    
    prompt = f'''Extract data from the document according to this JSON schema:
{json.dumps(schema, indent=2)}

Document:
{doc}

Return only valid JSON matching the schema.'''
    
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'},
        temperature=0
    )
    
    # Parse and validate with Pydantic
    data = json.loads(resp.choices[0].message.content)
    return model(**data)

# Extract and validate
invoice = extract_pydantic(SAMPLE_INVOICE, Invoice)
print('Pydantic validated result:')
print(f'  Invoice Number: {invoice.invoice_number}')
print(f'  Vendor: {invoice.vendor}')
print(f'  Customer: {invoice.customer}')
print(f'  Total: ${invoice.total}')
print(f'  Date: {invoice.date}')

Pydantic validated result:
  Invoice Number: INV-2024-001
  Vendor: TechCorp Solutions
  Customer: Acme Inc
  Total: $6200.0
  Date: January 15, 2024


## 6. Building a Document Processor (~10 min)

Combine everything into a reusable processor that handles **multiple invoice formats**. 

The key insight: by using a well-defined schema and clear prompts, the LLM can extract the same structured data from vastly different document formats!

In [9]:
class DocumentProcessor:
    """A reusable document data extractor."""
    
    def __init__(self, client, model='gpt-4o-mini'):
        self.client = client
        self.model = model
    
    def extract(self, doc, pydantic_model):
        """Extract structured data from a document."""
        schema = pydantic_model.model_json_schema()
        
        prompt = f'''You are a document data extraction assistant.
Extract all relevant fields from the document according to this schema:
{json.dumps(schema, indent=2)}

Document:
{doc}

Return only valid JSON. Use null for missing fields.'''
        
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=[{'role': 'user', 'content': prompt}],
            response_format={'type': 'json_object'},
            temperature=0
        )
        
        data = json.loads(resp.choices[0].message.content)
        return pydantic_model(**data)

# Create processor
processor = DocumentProcessor(client)

# Process ALL invoices from different formats
md("## 📊 Processing All Invoices\n\nExtracting structured data from **6 different invoice formats**...\n\n---")

results = []
for name, content in INVOICES.items():
    try:
        invoice = processor.extract(content, Invoice)
        results.append({
            'source': name,
            'invoice_number': invoice.invoice_number,
            'vendor': invoice.vendor,
            'customer': invoice.customer,
            'total': invoice.total,
            'date': invoice.date,
            'status': '✅'
        })
    except Exception as e:
        results.append({
            'source': name,
            'invoice_number': 'ERROR',
            'vendor': str(e)[:30],
            'customer': '-',
            'total': 0,
            'date': '-',
            'status': '❌'
        })

# Display results as markdown table
table = "| Status | Source File | Invoice # | Vendor | Customer | Total | Date |\n"
table += "|:------:|-------------|-----------|--------|----------|------:|------|\n"

for r in results:
    total_str = f"${r['total']:,.2f}" if isinstance(r['total'], (int, float)) else str(r['total'])
    table += f"| {r['status']} | {r['source'][:25]} | {r['invoice_number']} | {r['vendor'][:20]} | {r['customer'][:15]} | {total_str} | {r['date']} |\n"

md(table)

# Summary
total_extracted = sum(r['total'] for r in results if r['status'] == '✅')
md(f"\n---\n\n### 💰 Summary\n\n- **Invoices processed:** {len(results)}\n- **Successful extractions:** {sum(1 for r in results if r['status'] == '✅')}\n- **Total value extracted:** ${total_extracted:,.2f}")

## 📊 Processing All Invoices

Extracting structured data from **6 different invoice formats**...

---

| Status | Source File | Invoice # | Vendor | Customer | Total | Date |
|:------:|-------------|-----------|--------|----------|------:|------|
| ✅ | invoice_002_simple_receip | REC-8847 | CloudHost Services | DataFlow Analyt | $406.00 | March 22, 2024 |
| ✅ | invoice_003_email_style | DP-2024-156 | DesignPro Studio | RetailMax | $15,000.00 | April 5, 2024 |
| ✅ | invoice_004_european_form | MS-2024-0892 | Müller & Schmidt Gmb | Nordic Tech AS | $11,959.50 | 2024-06-18 |
| ✅ | invoice_005_handwritten_s | PP-5521 | Pete's Plumbing Serv | Mr. James Wilso | $355.00 | 2024-05-10 |
| ✅ | invoice_006_minimal | QK-99012 | QuickShip Logistics | FreshMart Groce | $65.00 | 2024-07-28 |



---

### 💰 Summary

- **Invoices processed:** 5
- **Successful extractions:** 5
- **Total value extracted:** $27,785.50

## 🎉 Summary

You learned four extraction strategies and saw them work on **6 different invoice formats**:

| Method | Pros | Cons |
|--------|------|------|
| **Zero-Shot** | Simple, no setup | Inconsistent format |
| **Few-Shot** | Better consistency | Requires examples |
| **JSON Mode** | Guaranteed valid JSON | Schema in prompt |
| **Pydantic** | Type validation, IDE support | More setup |

### Invoice Formats Processed
- 📋 **Corporate** - Formal business invoice with headers
- 🧾 **Receipt** - Simple receipt format
- 📧 **Email** - Invoice embedded in email text
- 🇪🇺 **European** - German format with different date style
- ✍️ **Handwritten** - Casual service invoice
- 📝 **Minimal** - Single-line compact format

### Key Insight
The LLM successfully extracted the **same structured schema** from vastly different formats - this is the power of prompt engineering for data extraction!

### Best Practices
- Use JSON mode for reliable parsing
- Use Pydantic for type safety and validation
- Provide few-shot examples for complex extractions
- Handle missing fields gracefully with Optional types
- Test your extractor on diverse document formats